# Electric Motor Temperature: 1D-CNN Regression (PyTorch)
Predicts the **permanent-magnet (rotor) temperature `pm`** from a sliding window of motor signals.
Dataset: https://www.kaggle.com/wkirgsn/electric-motor-temperature

**What was fixed / improved vs. the original notebook** (details in the README at the end):
- Windows built per profile with a vectorized sliding window (no leakage across profiles, ~all windows used instead of 1/seq_len).
- Split by *profile* and scaler fit on *train only* (the original random split of overlapping windows + global scaling leaked test info).
- Crashes fixed: undefined names, wrong class names, wrong kwargs, shadowed `optim`, bad `test()` loop, loading a checkpoint that was never saved.
- Faster training: big batches, `TensorDataset`, `no_grad` eval, AMP on GPU, LR scheduler, early stopping, best-checkpoint saving.

In [ ]:
import os, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 2
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Configuration

In [ ]:
CONFIG = dict(
    seq_len     = 5,       # window length (time steps)
    horizon     = 0,       # 0 = predict target at last step of window, 1 = next step, ...
    batch_size  = 256,
    lr          = 1e-3,
    epochs      = 50,
    patience    = 8,       # early-stopping patience (epochs)
    val_profiles  = 2,     # number of profiles held out for validation
    test_profiles = 2,     # number of profiles held out for final testing
    ckpt_path   = "./best_model.pt",
)

TARGET = "pm"
# Other temperatures + torque are outputs of the motor, so they are NOT used as inputs.
EXCLUDE = ["pm", "torque", "stator_yoke", "stator_tooth", "stator_winding", "profile_id"]

## 2. Load data

In [ ]:
CANDIDATES = [
    "/kaggle/input/electric-motor-temperature/pmsm_temperature_data.csv",
    "../input/electric-motor-temperature/pmsm_temperature_data.csv",
    "./pmsm_temperature_data.csv",
]
csv_path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        "Dataset not found. Download it from https://www.kaggle.com/wkirgsn/electric-motor-temperature "
        "and place pmsm_temperature_data.csv next to this notebook."
    )

df = pd.read_csv(csv_path)
df["profile_id"] = df["profile_id"].astype("int16")   # keep numeric; 'category' only complicates filtering
feature_list = [c for c in df.columns if c not in EXCLUDE]

print(f"{len(df):,} rows | {df.profile_id.nunique()} profiles")
print("Features:", feature_list)
print("Missing values:", int(df.isna().sum().sum()))
df.head()

## 3. Split by profile
Each `profile_id` is one continuous measurement session. Splitting *by profile* means train/val/test never share
overlapping windows, giving an honest estimate of generalisation to unseen drives.

In [ ]:
rng = np.random.default_rng(SEED)
prof_ids = rng.permutation(df.profile_id.unique())

n_test, n_val = CONFIG["test_profiles"], CONFIG["val_profiles"]
test_ids  = prof_ids[:n_test]
val_ids   = prof_ids[n_test:n_test + n_val]
train_ids = prof_ids[n_test + n_val:]
print("train:", sorted(train_ids.tolist()))
print("val  :", sorted(val_ids.tolist()))
print("test :", sorted(test_ids.tolist()))

# Scale using TRAIN data only (no leakage). Features and target scaled separately so we can invert the target.
train_df = df[df.profile_id.isin(train_ids)]
x_scaler = MinMaxScaler().fit(train_df[feature_list])
y_scaler = MinMaxScaler().fit(train_df[[TARGET]])

## 4. Build windows (vectorized)

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def build_seq(X, y, seq_len, horizon=0):
    """
    X: (T, F) array, y: (T,) array  -> windows of shape (N, F, seq_len) and targets (N, 1).
    Window i covers rows [i, i+seq_len); its target is y[i + seq_len - 1 + horizon].
    """
    n = len(X) - seq_len + 1 - horizon
    if n <= 0:
        return np.empty((0, X.shape[1], seq_len), np.float32), np.empty((0, 1), np.float32)
    w = sliding_window_view(X, seq_len, axis=0)[:n]          # (n, F, seq_len) view, no copy
    t = y[seq_len - 1 + horizon : seq_len - 1 + horizon + n]
    return np.ascontiguousarray(w, dtype=np.float32), t.astype(np.float32).reshape(-1, 1)

def make_split(profile_ids):
    Xs, ys = [], []
    for pid in profile_ids:
        part = df[df.profile_id == pid]                       # windows never cross profile boundaries
        X = x_scaler.transform(part[feature_list])
        y = y_scaler.transform(part[[TARGET]]).ravel()
        w, t = build_seq(X, y, CONFIG["seq_len"], CONFIG["horizon"])
        Xs.append(w); ys.append(t)
    return torch.from_numpy(np.concatenate(Xs)), torch.from_numpy(np.concatenate(ys))

X_train, y_train = make_split(train_ids)
X_val,   y_val   = make_split(val_ids)
X_test,  y_test  = make_split(test_ids)
print("train", tuple(X_train.shape), "| val", tuple(X_val.shape), "| test", tuple(X_test.shape))

In [ ]:
pin = device.type == "cuda"
bs = CONFIG["batch_size"]
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=bs, shuffle=True,  pin_memory=pin)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=bs*4, shuffle=False, pin_memory=pin)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=bs*4, shuffle=False, pin_memory=pin)

## 5. Model

In [ ]:
class CNNRegressor(nn.Module):
    def __init__(self, n_features, seq_len):
        super().__init__()
        k = min(3, seq_len)                                   # kernel can't exceed the window length
        self.features = nn.Sequential(
            nn.Conv1d(n_features, 16, kernel_size=k), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=1),         nn.ReLU(),
        )
        with torch.no_grad():                                 # infer flattened size once
            flat = self.features(torch.zeros(1, n_features, seq_len)).numel()
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 30), nn.ReLU(),
            nn.Linear(30, 1),
        )

    def forward(self, x):
        return self.head(self.features(x))

n_features = X_train.shape[1]
model = CNNRegressor(n_features, CONFIG["seq_len"]).to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 6. Training (AMP, LR scheduler, early stopping, best-checkpoint saving)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
use_amp = device.type == "cuda"
scaler_amp = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        total += criterion(net(xb), yb).item() * len(xb)
        n += len(xb)
    return total / n

def train(net, n_epochs, patience, interval=1):
    history = {"train": [], "val": []}
    best_val, best_state, bad = float("inf"), None, 0
    for epoch in range(1, n_epochs + 1):
        net.train()
        run, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(net(xb), yb)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
            run += loss.item() * len(xb); n += len(xb)

        tr, va = run / n, evaluate(net, val_loader)
        history["train"].append(tr); history["val"].append(va)
        scheduler.step(va)

        if va < best_val - 1e-7:
            best_val, bad = va, 0
            best_state = copy.deepcopy(net.state_dict())
            torch.save(best_state, CONFIG["ckpt_path"])
        else:
            bad += 1

        if epoch % interval == 0:
            print(f"Epoch {epoch:3d} | train {tr:.6f} | val {va:.6f} | lr {optimizer.param_groups[0]['lr']:.1e}")
        if bad >= patience:
            print(f"Early stopping at epoch {epoch} (best val {best_val:.6f})")
            break

    net.load_state_dict(best_state)
    return history

t0 = time.time()
history = train(model, CONFIG["epochs"], CONFIG["patience"])
print(f"Training time: {time.time() - t0:.1f}s")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["train"], label="Train")
plt.plot(history["val"],   label="Validation")
plt.title("Loss vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("MSE (scaled)")
plt.yscale("log"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Evaluation on held-out test profiles

In [ ]:
@torch.no_grad()
def get_pred(net, loader):
    net.eval()
    ts, os_ = [], []
    for xb, yb in loader:
        os_.append(net(xb.to(device)).cpu()); ts.append(yb)
    return torch.cat(ts).numpy(), torch.cat(os_).numpy()

# reload the best saved checkpoint (this is what the original notebook tried to do without ever saving one)
best = CNNRegressor(n_features, CONFIG["seq_len"]).to(device)
best.load_state_dict(torch.load(CONFIG["ckpt_path"], map_location=device))

t_s, p_s = get_pred(best, test_loader)
# back to degrees Celsius
targets, outputs = y_scaler.inverse_transform(t_s).ravel(), y_scaler.inverse_transform(p_s).ravel()

print(f"R2 : {r2_score(targets, outputs):.4f}")
print(f"MAE: {mean_absolute_error(targets, outputs):.3f} °C")
print(f"RMSE: {np.sqrt(mean_squared_error(targets, outputs)):.3f} °C")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(targets, outputs, s=4, alpha=0.4)
lims = [targets.min(), targets.max()]
ax[0].plot(lims, lims, "r--", lw=1)
ax[0].set(xlabel="Target (°C)", ylabel="Predicted (°C)", title="Predicted vs Target")

n = min(2000, len(targets))
ax[1].plot(targets[:n], label="Target", lw=1)
ax[1].plot(outputs[:n], label="Predicted", lw=1)
ax[1].set(xlabel="Time step", ylabel="PM temperature (°C)", title="First test samples over time")
ax[1].legend()
plt.tight_layout(); plt.show()

## README: bugs fixed
| # | Original problem | Fix |
|---|---|---|
| 1 | `target_list` undefined (only `t_list` existed) | single `TARGET = "pm"` |
| 2 | `build_seq(..., sequence_length=)` but parameter was `seq_length` → `TypeError` | rewritten, consistent names |
| 3 | `build_seq` looped over `rows / seq_length` (kept ~20% of windows) and indexed target at `i+seq_length+1` (off-by-one / `IndexError` at the end) | vectorized sliding window, correct target index, `horizon` option |
| 4 | `indices` undefined in train/test split | split by profile, no manual index math |
| 5 | `super(Network, self)` — class is called `Model` → `NameError` | `super().__init__()` |
| 6 | `get_lin_in_size()` called but method named `get_size()` | flattened size inferred once with a dummy forward pass |
| 7 | `sequence_length` undefined (variable was `seq_l`) | `CONFIG["seq_len"]` |
| 8 | `optim = optim.Adam(...)` and `loss = nn.MSELoss()` shadowed module / loss var; `train()` called with undefined `net`, `optimizer`, `criterion` | properly named `model`, `optimizer`, `criterion` |
| 9 | `test()` did `data, target = model.to(device), target.to(device)` (passed a *model* as input) | fixed; evaluation under `torch.no_grad()` and `eval()` |
| 10 | Checkpoint `model_single_measurement.pt` loaded but never saved | best checkpoint saved during training |
| 11 | Loss printed only when `epoch % interval == 1` | configurable interval |
| 12 | `get_pred` evaluated `model` on batch size 1 (very slow) | batched inference |

## README: methodology & efficiency improvements
- **No data leakage**: original scaled the whole profile before splitting and randomly split overlapping windows (neighbouring windows ≈ identical). Now: split by profile, scaler fit on train profiles only.
- **All profiles used** (original used only `profile_id == 6`).
- **Speed**: vectorized windows (`sliding_window_view`), pre-converted float32 tensors in `TensorDataset`, batch size 256 instead of 10, `shuffle=True` for training, AMP on GPU, pinned memory.
- **Better training**: LR scheduler, early stopping, best-model restore, validation set separate from test set.
- **Reporting**: R², MAE and RMSE in real °C, plus a time-series plot.
- Unused imports (`seaborn`, `F`) removed; dataset path made robust outside Kaggle.